In [ ]:
from tryHOI import spikes
%load_ext autoreload
%autoreload 2
import numpy as np
import fmatoolbox as fma
import ISRUtilities as isru
import xarray as xr
import astropy
import pathlib
froot = pathlib.Path().cwd().parent.parent / 'Results/Figures/ISIntervals'
batch_file = '/mnt/hubel-data-103/Pietro/InfraSlowNRPaper/Data/IS_intervals.batch'
do_save = False

In [ ]:
def _unitISAPhase(session,regs=None,when='sleep.*#0',n_bins=100,shuffle=0):

    R = fma.regions.regions(session,phases=when,events='InfraSlowRhythm/infraslowaval',load_spikes=False)
    regs = R.ids if regs is None else np.array(regs)[np.isin(regs,R.ids)]
    isa = R.eventIntervals('slownr')
    on = R.eventIntervals('slowavalnr')
    if on.size == 0:
        return None

    isa_phase, on_fraction = isru.ISPhase({0: on[:,0], 2*np.pi: isa[:,1], np.pi: on[:,1]},isa=isa)
    isa_phase_sorted = np.sort(isa_phase[:,1]) # estimation of time occupancy

    spikes = R.spikes(regs=regs)
    spikes = fma.general.restrict(spikes,isa) # keep only spikes in ISA

    # shuffle spikes
    if shuffle > 0:
        sss = fma.general.shuffleEvents(spikes,n=shuffle,intervals=isa)
        spikes = np.column_stack((spikes,sss))

    # parameters
    bins = np.linspace(0,2*np.pi,n_bins+1)
    centers = (bins[1:] + bins[:-1]) / 2
    # distribution of ISA phases for tuning curve
    isa_dist = np.full_like(centers,on_fraction)
    isa_dist[centers >= np.pi] = 1 - on_fraction
    isa_dist /= np.trapezoid(isa_dist,centers)

    #event_phase = {e: np.full_like(events[e],np.nan) for e in events}
    event_phase = {e: np.interp(events[e],isa_phase[:,0],isa_phase[:,1]) for e in events} # {event name: (event, shuffle)}
    dist = np.full((len(events),n_bins,shuffle+1),np.nan) # (event, phase, shuffle)
    dist_transformed = np.full((len(events),n_bins,shuffle+1),np.nan)
    p = np.full((len(events),2,shuffle+1),np.nan)
    for i, e in enumerate(events):
        #this_phases = []
        #if len(events[e]) > 0:
        #    this_phases = np.interp(events[e],isa_phase[:,0],isa_phase[:,1])
        #    event_phase[e] = this_phases
        for k in range(shuffle+1):
            this_phases = event_phase[e][:,k]
            d, _ = np.histogram(this_phases,bins=bins,density=True)
            dist[i,:,k] = d

            # CDF transform, let F(phi) be the empirical CDF of ISA phase
            u = np.searchsorted(isa_phase_sorted,this_phases,side="right") / len(isa_phase_sorted) # estimate CDF transform of each event's phase, in [0,1]
            p[i,:,k] = astropy.stats.kuiper(u)
            u = u * 2 * np.pi # transformed phase, uniform if events are uniform w.r.t. ISA phase
            dist_transformed[i,:,k] = np.histogram(u,bins=bins,density=True)[0]

    # make distributions walk the full circle
    centers = np.append(centers,centers[-1]+centers[1]-centers[0])
    tuning_curve = dist / isa_dist[None,:,None]
    dist = np.concatenate((dist,dist[:,0:1]),axis=1)
    tuning_curve = np.concatenate((tuning_curve,tuning_curve[:,0:1]),axis=1)
    dist_transformed = np.concatenate((dist_transformed,dist_transformed[:,0:1]),axis=1)

    dist = xr.DataArray(dist,dims=['ev','phi','shuf'],coords={'ev': list(events.keys()), 'phi': centers, 'shuf': [False]+[True]*shuffle, 'rat': int(R.rat)})
    tuning_curve = xr.DataArray(tuning_curve,dims=['ev','phi','shuf'],coords={'ev': list(events.keys()), 'phi': centers, 'shuf': [False]+[True]*shuffle, 'rat': int(R.rat)})
    dist_transformed = xr.DataArray(dist_transformed,dims=['ev','phi','shuf'],coords={'ev': list(events.keys()), 'phi': centers, 'shuf': [False]+[True]*shuffle, 'rat': int(R.rat)}) # 'p': ('event', p),
    p = xr.DataArray(p,dims=['ev','val','shuf'],coords={'ev': list(events.keys()), 'val': ['D','p'], 'shuf': [False]+[True]*shuffle, 'rat': int(R.rat)})

    return event_phase, dist, tuning_curve, dist_transformed, p, isa_phase